In [1]:
%pip install flask argon2-cffi flask-sqlalchemy


In [2]:
from flask import Flask, request, jsonify, session
from flask_sqlalchemy import SQLAlchemy
from argon2 import PasswordHasher
from datetime import datetime, timedelta, timezone
import secrets

In [3]:
app = Flask(__name__)

app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///auth.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

app.config["SECRET_KEY"] = secrets.token_hex(32)
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SECURE"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"
app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(minutes=30)

db = SQLAlchemy(app)

password_hasher = PasswordHasher(
    time_cost=2,
    memory_cost=19456,
    parallelism=1
)

print("Authentication setup completed successfully!")

Authentication setup completed successfully!


In [4]:
from flask import session, jsonify
from flask_wtf.csrf import CSRFProtect, generate_csrf

csrf = CSRFProtect()
csrf.init_app(app)

@app.route("/api/auth/csrf-token", methods=["GET"])
def get_csrf_token():
    return jsonify({"csrf_token": generate_csrf()})

print("CSRF protection and token endpoint configured!")

CSRF protection and token endpoint configured!


In [5]:
class User(db.Model):
    __tablename__ = "users"

    id = db.Column(db.Integer, primary_key=True)
    display_name = db.Column(db.String(50), nullable=False)
    email = db.Column(db.String(255), unique=True, nullable=False)
    password_hash = db.Column(db.String(255), nullable=False)
    birth_year = db.Column(db.Integer, nullable=False)
    weight = db.Column(db.Float, nullable=False)

    failed_login_attempts = db.Column(
        db.Integer, default=0, nullable=False
    )

    locked_until = db.Column(
        db.DateTime(timezone=True), nullable=True
    )

    privacy_notice_version = db.Column(
        db.String(20), nullable=False
    )

    privacy_accepted_at = db.Column(
        db.DateTime(timezone=True), nullable=False
    )


class SecurityLog(db.Model):
    __tablename__ = "security_logs"

    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, nullable=True)
    event_type = db.Column(db.String(50), nullable=False)

    timestamp = db.Column(
        db.DateTime(timezone=True),
        default=lambda: datetime.now(timezone.utc),
        nullable=False
    )


with app.app_context():
    db.create_all()

print("User and SecurityLog tables created successfully!")

User and SecurityLog tables created successfully!


In [6]:
def log_security_event(event_type, user_id=None):
    log = SecurityLog(
        user_id=user_id,
        event_type=event_type
    )

    db.session.add(log)
    db.session.commit()

    print("Security event logged:", event_type)

In [7]:
from argon2.exceptions import HashingError

@app.route("/api/auth/register", methods=["POST"])
def register():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    required = [
        "display_name", "email", "password",
        "birth_year", "weight", "privacy_consent"
    ]

    if any(field not in data for field in required):
        return jsonify({"error": "Missing required fields"}), 400

    name = data["display_name"]
    email = data["email"]
    password = data["password"]
    birth_year = data["birth_year"]
    weight = data["weight"]
    consent = data["privacy_consent"]

    # Validate display name
    if not isinstance(name, str) or not 1 <= len(name.strip()) <= 50:
        return jsonify({"error": "Invalid display name"}), 400

    # Normalize email
    if isinstance(email, str):
        email = email.strip().lower()

    # Validate email
    if not isinstance(email, str) or not (
        3 <= len(email) <= 255
        and email.count("@") == 1
        and "." in email.split("@")[-1]
        and not any(c.isspace() for c in email)
    ):
        return jsonify({"error": "Invalid email"}), 400

    # Validate password
    if not isinstance(password, str) or not 8 <= len(password) <= 128:
        return jsonify({
            "error": "Password must be 8-128 characters"
        }), 400

    # Validate birth year
    current_year = datetime.now(timezone.utc).year

    if (
        type(birth_year) is not int
        or not current_year - 100 <= birth_year <= current_year - 18
    ):
        return jsonify({"error": "Invalid birth year"}), 400

    # Validate weight
    if (
        type(weight) not in (int, float)
        or not 20 <= weight <= 300
    ):
        return jsonify({"error": "Invalid weight"}), 400

    # Validate privacy consent
    if consent is not True:
        return jsonify({
            "error": "Privacy consent is required"
        }), 400

    # Check duplicate email
    if User.query.filter_by(email=email).first():
        return jsonify({
            "error": "Email already registered"
        }), 409

    # Hash password using Argon2id
    try:
        hashed_password = password_hasher.hash(password)

    except HashingError:
        return jsonify({
            "error": "Registration failed"
        }), 500

    # Create user
    user = User(
        display_name=name.strip(),
        email=email,
        password_hash=hashed_password,
        birth_year=birth_year,
        weight=weight,
        failed_login_attempts=0,
        privacy_notice_version="1.0",
        privacy_accepted_at=datetime.now(timezone.utc)
    )

    db.session.add(user)
    db.session.commit()

    return jsonify({
        "message": "Account registered successfully",
        "user_id": user.id
    }), 201

In [8]:
from argon2.exceptions import VerifyMismatchError, VerificationError

@app.route("/api/auth/login", methods=["POST"])
def login():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    email = data.get("email")
    password = data.get("password")

    if not isinstance(email, str) or not isinstance(password, str):
        return jsonify({"error": "Email and password are required"}), 400

    email = email.strip().lower()
    user = User.query.filter_by(email=email).first()

    if user is None:
        log_security_event("LOGIN_FAILED")
        return jsonify({"error": "Invalid email or password"}), 401

    now = datetime.now(timezone.utc)
    locked_until = user.locked_until

    if locked_until is not None:
        if locked_until.tzinfo is None:
            locked_until = locked_until.replace(tzinfo=timezone.utc)

        if now < locked_until:
            log_security_event("LOGIN_BLOCKED", user.id)
            return jsonify({
                "error": "Account temporarily locked. Try again later."
            }), 423

        user.locked_until = None
        user.failed_login_attempts = 0
        db.session.commit()

    try:
        password_hasher.verify(user.password_hash, password)

    except (VerifyMismatchError, VerificationError):
        user.failed_login_attempts += 1

        if user.failed_login_attempts >= 5:
            user.locked_until = now + timedelta(minutes=15)

        db.session.commit()

        log_security_event("LOGIN_FAILED", user.id)

        return jsonify({
            "error": "Invalid email or password"
        }), 401

    user.failed_login_attempts = 0
    user.locked_until = None
    db.session.commit()

    session.clear()
    session["user_id"] = user.id
    session.permanent = True

    app.session_interface.regenerate(session)

    log_security_event("LOGIN_SUCCESS", user.id)

    return jsonify({
        "message": "Login successful"
    }), 200

In [9]:
@app.route("/api/auth/logout", methods=["POST"])
def logout():
    user_id = session.get("user_id")

    if user_id is None:
        return jsonify({
            "error": "Authentication required"
        }), 401

    log_security_event("LOGOUT", user_id)

    session.clear()

    return jsonify({
        "message": "Logout successful"
    }), 200

In [10]:
@app.route("/api/auth/change-credentials", methods=["POST"])
def change_credentials():
    user_id = session.get("user_id")

    if user_id is None:
        return jsonify({"error": "Authentication required"}), 401

    user = db.session.get(User, user_id)

    if user is None:
        session.clear()
        return jsonify({"error": "Authentication required"}), 401

    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid JSON"}), 400

    current_password = data.get("current_password")

    if not isinstance(current_password, str):
        return jsonify({"error": "Current password is required"}), 400

    try:
        password_hasher.verify(user.password_hash, current_password)
    except (VerifyMismatchError, VerificationError):
        return jsonify({"error": "Incorrect current password"}), 403

    new_email = data.get("new_email")
    new_password = data.get("new_password")

    if new_email is None and new_password is None:
        return jsonify({"error": "No changes requested"}), 400

    if new_email is not None:
        if not isinstance(new_email, str):
            return jsonify({"error": "Invalid email"}), 400

        new_email = new_email.strip().lower()

        if (
            len(new_email) > 255
            or new_email.count("@") != 1
            or "." not in new_email.split("@")[-1]
            or any(c.isspace() for c in new_email)
        ):
            return jsonify({"error": "Invalid email"}), 400

        existing = User.query.filter_by(email=new_email).first()

        if existing and existing.id != user.id:
            return jsonify({"error": "Email already registered"}), 409

        user.email = new_email

    if new_password is not None:
        if not isinstance(new_password, str) or not 8 <= len(new_password) <= 128:
            return jsonify({
                "error": "Password must be 8-128 characters"
            }), 400

        user.password_hash = password_hasher.hash(new_password)

    db.session.commit()

    return jsonify({
        "message": "Credentials updated successfully"
    }), 200

In [11]:
%pip install Flask-Session

In [12]:
from flask_session import Session

app.config["SESSION_TYPE"] = "cachelib"
app.config["SESSION_SERIALIZATION_FORMAT"] = "json"
app.config["SESSION_PERMANENT"] = True
app.config["SESSION_REFRESH_EACH_REQUEST"] = True

Session(app)

print("Server-side sessions configured successfully!")

Server-side sessions configured successfully!


/usr/local/lib/python3.13/dist-packages/flask_session/cachelib/cachelib.py:43: RuntimeWarning: No valid cachelib instance provided, attempting to create a new instance on localhost with default settings.
  warnings.warn(


In [13]:
with app.test_client() as client:
    response = client.post("/api/auth/register", json={
        "display_name": "Test User",
        "email": "test@example.com",
        "password": "TestPass123!",
        "birth_year": 2000,
        "weight": 60,
        "privacy_consent": True
    })

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 400
Response: None


In [14]:
with app.test_client() as client:
    response = client.post("/api/auth/login", json={
        "email": "test@example.com",
        "password": "TestPass123!"
    })

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 400
Response: None


In [15]:
with app.test_client() as client:

    # Login first
    client.post("/api/auth/login", json={
        "email": "test@example.com",
        "password": "TestPass123!"
    })

    # Logout
    response = client.post("/api/auth/logout")

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 400
Response: None


In [16]:
with app.test_client() as client:
    for i in range(5):
        response = client.post("/api/auth/login", json={
            "email": "test@example.com",
            "password": "WrongPassword123"
        })
        print(f"Attempt {i+1}: {response.status_code}")

    response = client.post("/api/auth/login", json={
        "email": "test@example.com",
        "password": "TestPass123!"
    })

    print("Login after 5 failures:", response.status_code)
    print("Response:", response.get_json())

Attempt 1: 400
Attempt 2: 400
Attempt 3: 400
Attempt 4: 400
Attempt 5: 400
Login after 5 failures: 400
Response: None


In [17]:
with app.test_client() as client:
    response = client.post(
        "/api/auth/change-credentials",
        json={"new_email": "new@example.com"}
    )

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 400
Response: None


In [18]:
with app.test_client() as client:

    # Create a new test account
    register_response = client.post("/api/auth/register", json={
        "display_name": "Test User Two",
        "email": "test2@example.com",
        "password": "OldPass123!",
        "birth_year": 2000,
        "weight": 60,
        "privacy_consent": True
    })

    # Login
    login_response = client.post("/api/auth/login", json={
        "email": "test2@example.com",
        "password": "OldPass123!"
    })

    # Change password
    response = client.post("/api/auth/change-credentials", json={
        "current_password": "OldPass123!",
        "new_password": "NewPass123!"
    })

    print("Login:", login_response.status_code)
    print("Change password:", response.status_code)
    print("Response:", response.get_json())

Login: 400
Change password: 400
Response: None


In [19]:
with app.test_client() as client:

    old_login = client.post("/api/auth/login", json={
        "email": "test2@example.com",
        "password": "OldPass123!"
    })

    new_login = client.post("/api/auth/login", json={
        "email": "test2@example.com",
        "password": "NewPass123!"
    })

    print("Old password:", old_login.status_code)
    print("New password:", new_login.status_code)

Old password: 400
New password: 400


In [20]:
from datetime import timedelta

print("Session timeout:", app.permanent_session_lifetime)
print("HttpOnly:", app.config["SESSION_COOKIE_HTTPONLY"])
print("Secure:", app.config["SESSION_COOKIE_SECURE"])
print("Session type:", app.config["SESSION_TYPE"])

Session timeout: 0:30:00
HttpOnly: True
Secure: True
Session type: cachelib


In [21]:
from datetime import datetime, timedelta, timezone
from unittest.mock import patch

with app.test_client() as client:
    login_response = client.post("/api/auth/login", json={
        "email": "test2@example.com",
        "password": "NewPass123!"
    })

    print("Login:", login_response.status_code)

    with client.session_transaction() as sess:
        print("Session active:", "user_id" in sess)
        print("Session lifetime:", app.permanent_session_lifetime)

Login: 400
Session active: False
Session lifetime: 0:30:00


In [22]:
with app.test_client() as client:

    login = client.post("/api/auth/login", json={
        "email": "test2@example.com",
        "password": "NewPass123!"
    })

    with client.session_transaction() as sess:
        before_logout = "user_id" in sess

    logout = client.post("/api/auth/logout")

    with client.session_transaction() as sess:
        after_logout = "user_id" in sess

    protected = client.post(
        "/api/auth/change-credentials",
        json={"current_password": "NewPass123!", "new_email": "new@example.com"}
    )

    print("Login:", login.status_code)
    print("Session before logout:", before_logout)
    print("Logout:", logout.status_code)
    print("Session after logout:", after_logout)
    print("Protected request after logout:", protected.status_code)

Login: 400
Session before logout: False
Logout: 400
Session after logout: False
Protected request after logout: 400


In [23]:
%pip install Flask-WTF

In [24]:
with app.test_client() as client:

    token_response = client.get("/api/auth/csrf-token")
    csrf_token = token_response.get_json()["csrf_token"]

    response = client.post(
        "/api/auth/register",
        json={
            "display_name": "CSRF Test User",
            "email": "csrf-test@example.com",
            "password": "TestPass123!",
            "birth_year": 2000,
            "weight": 60,
            "privacy_consent": True
        },
        headers={"X-CSRFToken": csrf_token}
    )

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 409
Response: {'error': 'Email already registered'}


In [25]:
with app.test_client() as client:

    token_response = client.get("/api/auth/csrf-token")
    csrf_token = token_response.get_json()["csrf_token"]

    response = client.post(
        "/api/auth/login",
        json={
            "email": "csrf-test@example.com",
            "password": "TestPass123!"
        },
        headers={"X-CSRFToken": csrf_token}
    )

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Security event logged: LOGIN_SUCCESS
Status: 200
Response: {'message': 'Login successful'}


In [26]:
with app.test_client() as client:

    response = client.post(
        "/api/auth/login",
        json={
            "email": "csrf-test@example.com",
            "password": "TestPass123!"
        }
    )

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 400
Response: None


In [27]:
with app.test_client() as client:

    # Get CSRF token
    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    # Test Login
    login_response = client.post(
        "/api/auth/login",
        json={
            "email": "csrf-test@example.com",
            "password": "TestPass123!"
        },
        headers={"X-CSRFToken": token}
    )

    print("Login Status:", login_response.status_code)
    print("Login Response:", login_response.get_json())

    # Get a fresh CSRF token after login
    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    # Test Logout
    logout_response = client.post(
        "/api/auth/logout",
        headers={"X-CSRFToken": token}
    )

    print("Logout Status:", logout_response.status_code)
    print("Logout Response:", logout_response.get_json())

    # Check Security Logs
    with app.app_context():
        logs = SecurityLog.query.order_by(
            SecurityLog.id.desc()
        ).limit(5).all()

        print("\nSecurity Logs:")
        for log in logs:
            print(log.event_type, log.user_id)

Security event logged: LOGIN_SUCCESS
Login Status: 200
Login Response: {'message': 'Login successful'}
Security event logged: LOGOUT
Logout Status: 200
Logout Response: {'message': 'Logout successful'}

Security Logs:
LOGOUT 3
LOGIN_SUCCESS 3
LOGIN_SUCCESS 3
LOGIN_SUCCESS 3
LOGIN_SUCCESS 3


In [28]:
with app.test_client() as client:

    # Get CSRF token
    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    # Login with incorrect password
    response = client.post(
        "/api/auth/login",
        json={
            "email": "csrf-test@example.com",
            "password": "WrongPassword123!"
        },
        headers={"X-CSRFToken": token}
    )

    print("Login Status:", response.status_code)
    print("Login Response:", response.get_json())

    # Check latest security log
    with app.app_context():
        latest_log = SecurityLog.query.order_by(
            SecurityLog.id.desc()
        ).first()

        print("Security Log:", latest_log.event_type)

Security event logged: LOGIN_FAILED
Login Status: 401
Login Response: {'error': 'Invalid email or password'}
Security Log: LOGIN_FAILED


In [29]:
with app.test_client() as client:

    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    before_cookie = client.get_cookie("session")
    before_sid = before_cookie.value if before_cookie else None

    response = client.post(
        "/api/auth/login",
        json={
            "email": "csrf-test@example.com",
            "password": "TestPass123!"
        },
        headers={"X-CSRFToken": token}
    )

    after_cookie = client.get_cookie("session")
    after_sid = after_cookie.value if after_cookie else None

    print("Login Status:", response.status_code)
    print("Session ID changed:", before_sid != after_sid)
    print("Session exists:", after_sid is not None)

Security event logged: LOGIN_SUCCESS
Login Status: 200
Session ID changed: True
Session exists: True


In [30]:
print("Session lifetime:",
      app.config["PERMANENT_SESSION_LIFETIME"])

print("Session refresh:",
      app.config["SESSION_REFRESH_EACH_REQUEST"])

print("Cookie HttpOnly:",
      app.config["SESSION_COOKIE_HTTPONLY"])

print("Cookie Secure:",
      app.config["SESSION_COOKIE_SECURE"])

Session lifetime: 0:30:00
Session refresh: True
Cookie HttpOnly: True
Cookie Secure: True


In [31]:
from datetime import timedelta
from flask import session

original_lifetime = app.config["PERMANENT_SESSION_LIFETIME"]

try:
    # Short timeout for testing only
    app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(seconds=1)

    with app.test_client() as client:

        token = client.get(
            "/api/auth/csrf-token"
        ).get_json()["csrf_token"]

        response = client.post(
            "/api/auth/login",
            json={
                "email": "csrf-test@example.com",
                "password": "TestPass123!"
            },
            headers={"X-CSRFToken": token}
        )

        print("Login Status:", response.status_code)

        if response.status_code == 200:
            import time
            time.sleep(2)

            protected_response = client.post(
                "/api/auth/change-credentials",
                json={},
                headers={"X-CSRFToken": token}
            )

            print("After timeout:", protected_response.status_code)
            print("Response:", protected_response.get_json())

finally:
    # Restore the required 30-minute timeout
    app.config["PERMANENT_SESSION_LIFETIME"] = original_lifetime

Security event logged: LOGIN_SUCCESS
Login Status: 200
After timeout: 400
Response: None


In [32]:
from datetime import timedelta
import time

original_lifetime = app.config["PERMANENT_SESSION_LIFETIME"]

try:
    app.config["PERMANENT_SESSION_LIFETIME"] = timedelta(seconds=1)

    with app.test_client() as client:

        token = client.get(
            "/api/auth/csrf-token"
        ).get_json()["csrf_token"]

        login_response = client.post(
            "/api/auth/login",
            json={
                "email": "csrf-test@example.com",
                "password": "TestPass123!"
            },
            headers={"X-CSRFToken": token}
        )

        print("Login Status:", login_response.status_code)

        time.sleep(2)

        # Check session directly without CSRF interference
        with client.session_transaction() as test_session:
            user_id = test_session.get("user_id")

        print("User ID after timeout:", user_id)
        print("Session expired:", user_id is None)

finally:
    app.config["PERMANENT_SESSION_LIFETIME"] = original_lifetime

Security event logged: LOGIN_SUCCESS
Login Status: 200
User ID after timeout: None
Session expired: True


In [33]:
with app.test_client() as client:

    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    response = client.post(
        "/api/auth/change-credentials",
        json={},
        headers={"X-CSRFToken": token}
    )

    print("Status:", response.status_code)
    print("Response:", response.get_json())

Status: 401
Response: {'error': 'Authentication required'}


In [34]:
import secrets

test_email = f"final-{secrets.token_hex(4)}@example.com"

with app.test_client() as client:

    # Get CSRF token
    token = client.get(
        "/api/auth/csrf-token"
    ).get_json()["csrf_token"]

    # Test registration with uppercase email and spaces
    response = client.post(
        "/api/auth/register",
        json={
            "display_name": "Final Test",
            "email": "  " + test_email.upper() + "  ",
            "password": "TestPass123!",
            "birth_year": 2000,
            "weight": 60,
            "privacy_consent": True
        },
        headers={"X-CSRFToken": token}
    )

    print("Register Status:", response.status_code)
    print("Register Response:", response.get_json())

    # Verify email was normalized and saved correctly
    with app.app_context():
        user = User.query.filter_by(email=test_email).first()
        print("Email saved correctly:", user is not None)

Register Status: 201
Register Response: {'message': 'Account registered successfully', 'user_id': 4}
Email saved correctly: True
